# 05 - 零件任务特定的 Z 校准

对每类零件的抓取点 Z 估计固定偏差，优先拟合可解释且稳定的 `offset_part`。

手眼坐标约定与 `D:\HIWI\Hand_eye_calibration.ipynb` 完全一致：

```text
p_base = T_base_gripper @ T_gripper_camera @ p_camera
```

本 notebook 内部保留兼容别名：`T_base_ee == T_base_gripper`，
`T_ee_cam == T_gripper_camera`。

**必须手动填写/提供**：所有零件真实高度、当前 `T_base_gripper`、抓取点像素；
并确认当前机器人 gripper/EE frame 与手眼标定采集时相同。

> **运行方式**：从上到下逐个运行。深度数据内部统一使用米（m），统计结果同时显示毫米（mm）。


## Block 1：导入库和参数设置

配置输入文件、patch、深度修正和 base Z 方向。


In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "cam_calibration").is_dir():
            return candidate
    raise RuntimeError("无法定位项目根目录；请从 Xinyu 项目内启动 Jupyter。")

PROJECT_ROOT = find_project_root()
CALIBRATION_ROOT = PROJECT_ROOT / "cam_calibration"
INPUT_DIR = CALIBRATION_ROOT / "inputs"
OUTPUT_ROOT = CALIBRATION_ROOT / "outputs"
SHARED_DIR = OUTPUT_ROOT / "shared"
CONFIG_DIR = CALIBRATION_ROOT / "config"
for directory in (INPUT_DIR, OUTPUT_ROOT, SHARED_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

patch_size=7
output_dir=OUTPUT_ROOT / "05_part_z_calibration"
output_dir.mkdir(parents=True,exist_ok=True)
use_depth_correction=True
height_sign=+1  # base +Z 向上用 +1；反之 -1

depth_calibration_path=SHARED_DIR / "depth_calibration_linear.json"
system_geometry_path=SHARED_DIR / "system_geometry.json"
hand_eye_result_path=CONFIG_DIR / "hand_eye_result.json"
intrinsics_path=SHARED_DIR / "camera_intrinsics.json"
part_heights_path=CONFIG_DIR / "part_heights.json"

if patch_size%2==0 or height_sign not in (-1,+1):
    raise ValueError("检查 patch_size 和 height_sign。")


## Block 2：加载内参、depth calibration、手眼和工作台平面

若统一 JSON 不存在，可在本 cell 的 fallback 变量中手动填写。


In [ ]:
manual_intrinsics={"fx":None,"fy":None,"cx":None,"cy":None}  # TODO fallback
manual_T_gripper_camera=np.eye(4)  # TODO fallback
manual_n_base=np.array([0.,0.,1.]) # TODO fallback
manual_d=0.0

def normalize_intrinsics_keys(data):
    data=dict(data)
    if "cx" not in data and "ppx" in data: data["cx"]=data["ppx"]
    if "cy" not in data and "ppy" in data: data["cy"]=data["ppy"]
    return data

if system_geometry_path.exists():
    geometry=json.loads(system_geometry_path.read_text(encoding="utf-8"))
    intrinsics=normalize_intrinsics_keys(geometry["camera_intrinsics"])
    matrix_data=geometry.get("T_gripper_camera",geometry.get("T_ee_cam"))
    if matrix_data is None: raise KeyError("system_geometry.json 缺少 T_gripper_camera/T_ee_cam")
    T_gripper_camera=np.asarray(matrix_data,float)
    n_base=np.asarray(geometry["plane"]["n_base"],float)
    d=float(geometry["plane"]["d"])
else:
    if intrinsics_path.exists():
        intrinsics=normalize_intrinsics_keys(
            json.loads(intrinsics_path.read_text(encoding="utf-8"))
        )
    else:
        intrinsics=manual_intrinsics
    if hand_eye_result_path.exists():
        hand_eye=json.loads(hand_eye_result_path.read_text(encoding="utf-8"))
        if hand_eye.get("T_gripper_camera") is None:
            raise RuntimeError(
                f"{hand_eye_result_path} 还是模板；请写入真实 4x4 T_gripper_camera。"
            )
        T_gripper_camera=np.asarray(hand_eye["T_gripper_camera"],float)
    else:
        if np.allclose(manual_T_gripper_camera, np.eye(4)):
            raise FileNotFoundError(
                f"缺少真实手眼结果: {hand_eye_result_path}，且手动矩阵仍为单位矩阵。"
            )
        T_gripper_camera=manual_T_gripper_camera
    n_base=manual_n_base
    d=manual_d

T_ee_cam=T_gripper_camera
if T_gripper_camera.shape!=(4,4) or not np.allclose(T_gripper_camera[3],[0,0,0,1]):
    raise ValueError("T_gripper_camera 不是合法 4x4 齐次矩阵。")
if any(intrinsics.get(k) is None for k in ("fx","fy","cx","cy")):
    raise ValueError("请填写/加载完整相机内参。")

plane_norm=np.linalg.norm(n_base)
if plane_norm<1e-12: raise ValueError("工作台平面法向量不能为零。")
n_base=n_base/plane_norm
d=d/plane_norm

if use_depth_correction and depth_calibration_path.exists():
    cal=json.loads(depth_calibration_path.read_text(encoding="utf-8"))
    a,b=float(cal["a"]),float(cal["b"])
else:
    use_depth_correction=False;a,b=1.,0.

print("intrinsics:",intrinsics)
print("T_gripper_camera (= T_ee_cam) =\n",T_gripper_camera)
print("coordinate convention: p_base = T_base_gripper @ T_gripper_camera @ p_camera")
print("plane:",n_base,d)
print("depth correction:",use_depth_correction,a,b)


## Block 3：定义零件高度

把所有 `None` 替换为实测高度（米）。未填写时会停止，避免静默使用错误高度。


In [ ]:
if not part_heights_path.exists():
    raise FileNotFoundError(
        f"缺少零件高度配置: {part_heights_path}。请填写模板中的真实高度（米）。"
    )
part_heights=json.loads(part_heights_path.read_text(encoding="utf-8"))
required_parts={"base","Gear_compound","Gear_large","Gear_small","Screw","Shaft_large","Shaft_small"}
missing_keys=sorted(required_parts-set(part_heights))
if missing_keys:
    raise KeyError("零件高度配置缺少: "+", ".join(missing_keys))
missing=[name for name,value in part_heights.items() if value is None or value<=0]
if missing:
    raise ValueError("请先填写这些零件的真实高度 h_part_m: "+", ".join(missing))


## Block 4：采集或加载一帧 RGB-D

`source_mode='live'` 实时采集 aligned RGB-D；`'file'` 加载 color 与米制 depth。


In [ ]:
source_mode="file" # "live" 或 "file"
color_path=PROJECT_ROOT / "realsense_pic/realsense_color_20260609_112227.png"
depth_path=PROJECT_ROOT / "realsense_pic/realsense_depth_raw_20260609_112227.npy"
offline_depth_scale_to_m=0.001

def capture_aligned():
    import pyrealsense2 as rs
    pipeline=rs.pipeline()
    config=rs.config()
    config.enable_stream(rs.stream.color,640,480,rs.format.bgr8,30)
    config.enable_stream(rs.stream.depth,640,480,rs.format.z16,30)
    profile=pipeline.start(config)
    aligner=rs.align(rs.stream.color)
    try:
        for _ in range(30):
            pipeline.wait_for_frames()
        frames=aligner.process(pipeline.wait_for_frames())
        color_frame=frames.get_color_frame()
        depth_frame=frames.get_depth_frame()
        if not color_frame or not depth_frame:
            raise RuntimeError("没有获得 aligned RGB-D 帧。")
        color=np.asanyarray(color_frame.get_data())
        raw=np.asanyarray(depth_frame.get_data())
        scale=float(profile.get_device().first_depth_sensor().get_depth_scale())
        return color,raw.astype(np.float64)*scale
    finally:
        pipeline.stop()

if source_mode=="live":
    color_image,depth_m=capture_aligned()
elif source_mode=="file":
    color_image=cv2.imread(str(color_path))
    if color_image is None:
        raise FileNotFoundError(color_path)
    loaded=np.load(depth_path)
    if isinstance(loaded,np.lib.npyio.NpzFile):
        if "depth_m" in loaded:
            depth_m=loaded["depth_m"].astype(float)
        elif "depth_raw" in loaded:
            depth_m=loaded["depth_raw"].astype(float)*offline_depth_scale_to_m
        else:
            raise KeyError("NPZ 必须包含 depth_m 或 depth_raw。")
    else:
        depth_m=loaded.astype(float)*offline_depth_scale_to_m
    if depth_m.shape != color_image.shape[:2]:
        raise ValueError("离线 depth 必须已经 align 到 color，且尺寸完全一致。")
else:
    raise ValueError("source_mode 只能是 live 或 file。")

plt.figure(figsize=(10,7))
plt.imshow(cv2.cvtColor(color_image,cv2.COLOR_BGR2RGB))
plt.title("Current part RGB")
plt.axis("off")
plt.show()

## Block 5：输入抓取点 pixel 和零件类型

可手动输入或点击图像。每次更换图像/零件后重新运行本 cell。


In [ ]:
selection_mode="click" # "manual" 或 "click"
manual_px,manual_py=color_image.shape[1]//2,color_image.shape[0]//2
part_name="Gear_large" # TODO: 当前零件
if part_name not in part_heights: raise KeyError("未知 part_name")
if selection_mode=="manual":
    px,py=int(manual_px),int(manual_py)
else:
    fig,ax=plt.subplots(figsize=(10,7));ax.imshow(cv2.cvtColor(color_image,cv2.COLOR_BGR2RGB))
    ax.set_title("点击抓取点");clicked=plt.ginput(1,timeout=0);plt.close(fig)
    if not clicked:raise RuntimeError("没有选择点")
    px,py=map(lambda x:int(round(x)),clicked[0])
print(part_name,(px,py))


## Block 6：读取抓取点 depth

保存单像素用于对照，计算过滤 0/NaN 后的 patch median，并显示 patch。


In [ ]:
half=patch_size//2
if px-half<0 or py-half<0 or px+half>=depth_m.shape[1] or py+half>=depth_m.shape[0]:
    raise ValueError("抓取点太靠近边缘")
patch=depth_m[py-half:py+half+1,px-half:px+half+1]
valid=patch[np.isfinite(patch)&(patch>0)]
if valid.size==0:raise RuntimeError("patch 没有有效 depth")
D_meas=float(np.median(valid))
D_single=float(depth_m[py,px]) if depth_m[py,px]>0 else np.nan
vis=color_image.copy();cv2.rectangle(vis,(px-half,py-half),(px+half,py+half),(0,255,0),2)
cv2.drawMarker(vis,(px,py),(0,0,255),cv2.MARKER_CROSS,16,2)
plt.figure(figsize=(10,7));plt.imshow(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB));plt.axis("off")
plt.title(f"{part_name}: single={D_single:.4f} m, median={D_meas:.4f} m");plt.show()


## Block 7：计算 `P_cam` 和 `P_base`

填写与当前 RGB-D 帧同步的 `T_base_gripper`。如果数据来自 Franka/libfranka 的列主序
`O_T_EE`，使用 `np.asarray(O_T_EE).reshape(4, 4, order="F")`。
变换严格采用 `p_base = T_base_gripper @ T_gripper_camera @ p_camera`。


In [ ]:
current_pose_path=INPUT_DIR / "current_robot_pose.json"
if not current_pose_path.exists():
    raise FileNotFoundError(
        f"缺少当前 RGB-D 帧同步的机器人位姿: {current_pose_path}。"
        "请连接机械臂后填写模板；禁止使用单位矩阵占位。"
    )
pose_payload=json.loads(current_pose_path.read_text(encoding="utf-8"))
T_base_gripper=np.asarray(pose_payload["T_base_gripper"],dtype=float)
if T_base_gripper.shape!=(4,4) or np.allclose(T_base_gripper,np.eye(4)):
    raise ValueError("T_base_gripper 必须是真实的非占位 4x4 位姿。")
T_base_ee=T_base_gripper  # 兼容旧变量名
fx,fy,cx,cy=[float(intrinsics[k]) for k in ("fx","fy","cx","cy")]
def deproject(D):return np.array([(px-cx)/fx*D,(py-cy)/fy*D,D])
def transform(T,p):return (T@np.r_[p,1.])[:3]
D_corr=a*D_meas+b if use_depth_correction else D_meas
T_base_cam=T_base_gripper@T_gripper_camera
P_cam_meas,P_cam_corr=deproject(D_meas),deproject(D_corr)
P_base_meas=transform(T_base_cam,P_cam_meas)
P_base_corr=transform(T_base_cam,P_cam_corr)
print("P_base before:",P_base_meas)
print("P_base after :",P_base_corr)


## Block 8：计算真实顶部高度

在抓取点 base `x,y` 处求一般工作台平面的 Z，再叠加带方向的零件高度。要求 `n_base[2]` 不接近零。


In [ ]:
if abs(n_base[2])<1e-8:raise ValueError("工作台平面无法写成 z=f(x,y)")
h_part=float(part_heights[part_name])
# 修正前后 x,y 略有差异，分别计算对应的局部桌面高度。
def table_z(x,y):return -(n_base[0]*x+n_base[1]*y+d)/n_base[2]
z_top_before=table_z(P_base_meas[0],P_base_meas[1])+height_sign*h_part
z_top_after=table_z(P_base_corr[0],P_base_corr[1])+height_sign*h_part
print("Expected top Z before/corr XY:",z_top_before,z_top_after,"m")


## Block 9：计算并累计 Z residual

首次运行会创建 `z_samples`；每个新样本运行一次本 cell，随后更换图像/位姿继续采集。


In [ ]:
z_error_before=float(P_base_meas[2]-z_top_before)
z_error_after=float(P_base_corr[2]-z_top_after)
print(f"Z error before={z_error_before*1000:.3f} mm, after={z_error_after*1000:.3f} mm")
if "z_samples" not in globals():z_samples=[]
z_samples.append({"part_name":part_name,"px":px,"py":py,"D_single_m":D_single,
                  "D_meas_m":D_meas,"D_corr_m":D_corr,
                  "z_meas_before_m":float(P_base_meas[2]),"z_meas_after_m":float(P_base_corr[2]),
                  "z_top_m":float(z_top_after),"z_error_before_m":z_error_before,
                  "z_error_after_m":z_error_after})
samples_df=pd.DataFrame(z_samples)
def summarize(group):
    e=group["z_error_after_m"].to_numpy()*1000
    return pd.Series({"n":len(e),"mean_mm":e.mean(),"std_mm":e.std(),
                      "MAE_mm":np.mean(np.abs(e)),"RMSE_mm":np.sqrt(np.mean(e**2))})
part_summary = pd.DataFrame({
    name: summarize(group) for name, group in samples_df.groupby("part_name")
}).T
display(part_summary)


## Block 10：可视化

比较各零件修正前后误差、总体直方图，以及误差随测量距离的变化。


In [ ]:
if len(samples_df)==0:raise RuntimeError("尚无样本")
plot_df=samples_df.copy()
plot_df["before_mm"]=plot_df["z_error_before_m"]*1000
plot_df["after_mm"]=plot_df["z_error_after_m"]*1000
fig,axes=plt.subplots(1,3,figsize=(17,5))
grouped=plot_df.groupby("part_name")[["before_mm","after_mm"]].mean()
grouped.plot(kind="bar",ax=axes[0]);axes[0].set_ylabel("Mean Z error [mm]")
axes[1].hist(plot_df["before_mm"],bins=20,alpha=.6,label="before")
axes[1].hist(plot_df["after_mm"],bins=20,alpha=.6,label="after");axes[1].legend()
axes[2].scatter(plot_df["D_meas_m"],plot_df["before_mm"],label="before")
axes[2].scatter(plot_df["D_meas_m"],plot_df["after_mm"],label="after")
axes[2].axhline(0,color="k",ls="--");axes[2].set(xlabel="D_meas [m]",ylabel="Z error [mm]");axes[2].legend()
fig.tight_layout();fig.savefig(output_dir/"part_z_errors.png",dpi=150);plt.show()
cv2.imwrite(str(output_dir/f"latest_{part_name}_patch.png"),vis)


## Block 11：拟合 task-specific Z offset

定义 `z_corrected = z_meas_after + offset_part`，因此 `offset_part = mean(z_top-z_meas_after)`。


In [ ]:
offsets={}
for name,group in samples_df.groupby("part_name"):
    error=group["z_error_after_m"].to_numpy()
    offset=-float(np.mean(error))
    corrected_error=error+offset
    offsets[name]={"z_offset_m":offset,"sample_count":len(group),
                   "before_offset_mean_error_mm":float(np.mean(error)*1000),
                   "before_offset_std_mm":float(np.std(error)*1000),
                   "after_offset_RMSE_mm":float(np.sqrt(np.mean(corrected_error**2))*1000)}
display(pd.DataFrame(offsets).T)


## Block 12：保存 task-specific correction

保存逐样本 CSV/NPZ 和 `part_z_offsets.json`。


In [ ]:
samples_df.to_csv(output_dir/"part_z_samples.csv",index=False)
np.savez_compressed(output_dir/"part_z_samples.npz",
                    **{c:samples_df[c].to_numpy() for c in samples_df.columns if c!="part_name"},
                    part_name=samples_df["part_name"].to_numpy(dtype=str))
payload={"model":"z_corrected = z_meas_after + z_offset_m","units":"meter",
         "height_sign":height_sign,"part_heights_m":part_heights,"parts":offsets}
(output_dir/"part_z_offsets.json").write_text(
    json.dumps(payload,indent=2,ensure_ascii=False),encoding="utf-8")
print("已保存:",(output_dir/"part_z_offsets.json").resolve())


## Block 13：结论提示

固定均值偏差且标准差小，适合 offset；波动大则更像零件表面、边缘或材质问题。


In [ ]:
for name,item in offsets.items():
    std=item["before_offset_std_mm"];offset=item["z_offset_m"]*1000
    if item["sample_count"]<5:
        verdict="样本不足，建议至少采集 5-10 个不同位置/姿态。"
    elif std<2.0:
        verdict=f"误差较稳定，可使用 {offset:+.3f} mm 的 task-specific offset。"
    else:
        verdict="波动较大，更像零件表面/边缘/材质导致的局部 depth 误差；考虑 CAD 高度、颜色顶部、慢速下降或力控。"
    print(f"{name}: {verdict}")
